# 01 — Sample Ranked Players and Collect Match Histories

This notebook builds the raw analytical dataset for the project.

It does **not** detect play sessions or perform analysis. Those belong in later notebooks.

### Pipeline

1. Sample players from four broad rank groups: Challenger, Diamond, Gold, Bronze.
2. Use League-V4 for rank-based sampling.
3. Fetch each sampled player's most recent Ranked Solo/Duo matches with Match-V5.
4. Cache full match JSON locally so shared matches are never downloaded twice.
5. Extract one player-match row for each sampled player.
6. Save `data/processed/players.csv` and `data/processed/ranked_match_history.csv`.

The sampling uses a fixed random seed for reproducibility.


## 1. Imports and configuration

Your `.env` file should contain:

```text
RIOT_API_KEY=RGAPI-your-current-development-key
```

The defaults sample **10 players per rank group** and request **100 Ranked Solo/Duo matches per player**. With a development key this can take a while, so reduce either number while testing.


In [ ]:
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("RIOT_API_KEY")
if not API_KEY:
    raise ValueError("RIOT_API_KEY was not found. Add it to your .env file.")

PLATFORM = "euw1"
REGION = "europe"
PLATFORM_BASE = f"https://{PLATFORM}.api.riotgames.com"
REGION_BASE = f"https://{REGION}.api.riotgames.com"

QUEUE_TYPE = "RANKED_SOLO_5x5"
RANKED_SOLO_QUEUE_ID = 420
RANK_GROUPS = ["CHALLENGER", "DIAMOND", "GOLD", "BRONZE"]
PLAYERS_PER_GROUP = 10
DIVISIONS = ["I", "II", "III", "IV"]
PAGES_PER_DIVISION = 1
RANKED_MATCHES_PER_PLAYER = 100
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

CACHE_DIR = Path("data/cache/matches")
PROCESSED_DIR = Path("data/processed")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"X-Riot-Token": API_KEY}

print("Match cache:", CACHE_DIR.resolve())
print("Processed data:", PROCESSED_DIR.resolve())


## 2. Riot request helper

All API traffic goes through one helper. It spaces requests conservatively, respects `Retry-After` on HTTP 429, retries temporary Riot server errors, and raises normal HTTP errors instead of silently returning bad data.


In [ ]:
MIN_REQUEST_INTERVAL = 1.25
MAX_RETRIES = 5
_last_request_time = 0.0

def riot_get(url, params=None):
    global _last_request_time

    for attempt in range(1, MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_time
        if elapsed < MIN_REQUEST_INTERVAL:
            time.sleep(MIN_REQUEST_INTERVAL - elapsed)

        response = requests.get(url, headers=HEADERS, params=params, timeout=30)
        _last_request_time = time.monotonic()

        if response.status_code == 429:
            wait = float(response.headers.get("Retry-After", 10))
            print(f"Rate limited — waiting {wait:.1f}s")
            time.sleep(wait)
            continue

        if response.status_code in {500, 502, 503, 504}:
            wait = min(2 ** attempt, 30)
            print(f"Temporary Riot error {response.status_code} — retrying in {wait}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        return response.json()

    raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts: {url}")


## 3. League-V4 sampling functions

Challenger has its own League-V4 endpoint. Diamond, Gold, and Bronze use the tier/division entries endpoint. We pool candidates across divisions and deduplicate by PUUID before sampling.

Division is retained as metadata but the analysis group is the broader `rank_group`.


In [ ]:
def get_challenger_candidates():
    url = f"{PLATFORM_BASE}/lol/league/v4/challengerleagues/by-queue/{QUEUE_TYPE}"
    league = riot_get(url)
    candidates = []

    for entry in league["entries"]:
        candidates.append({
            "puuid": entry["puuid"],
            "rank_group": "CHALLENGER",
            "tier": "CHALLENGER",
            "division": entry.get("rank"),
            "league_points": entry.get("leaguePoints"),
            "wins": entry.get("wins"),
            "losses": entry.get("losses"),
        })

    return candidates


def get_tier_candidates(tier, divisions=DIVISIONS, pages=PAGES_PER_DIVISION):
    candidates = []

    for division in divisions:
        url = f"{PLATFORM_BASE}/lol/league/v4/entries/{QUEUE_TYPE}/{tier}/{division}"

        for page in range(1, pages + 1):
            entries = riot_get(url, params={"page": page})
            if not entries:
                break

            for entry in entries:
                candidates.append({
                    "puuid": entry["puuid"],
                    "rank_group": tier,
                    "tier": entry.get("tier", tier),
                    "division": entry.get("rank", division),
                    "league_points": entry.get("leaguePoints"),
                    "wins": entry.get("wins"),
                    "losses": entry.get("losses"),
                })

    by_puuid = {player["puuid"]: player for player in candidates}
    return list(by_puuid.values())


## 4. Build candidate pools

In [ ]:
candidate_pools = {}
candidate_pools["CHALLENGER"] = get_challenger_candidates()

for tier in ["DIAMOND", "GOLD", "BRONZE"]:
    candidate_pools[tier] = get_tier_candidates(tier)

for rank_group, candidates in candidate_pools.items():
    print(f"{rank_group:10s}: {len(candidates):4d} unique candidates")


## 5. Sample players

We sample independently within each broad rank group. The fixed `RANDOM_SEED` makes the selection deterministic for a given candidate pool.


In [ ]:
sampled_players = []

for rank_group in RANK_GROUPS:
    candidates = candidate_pools[rank_group]

    if len(candidates) < PLAYERS_PER_GROUP:
        raise ValueError(
            f"{rank_group} only has {len(candidates)} candidates, "
            f"but {PLAYERS_PER_GROUP} were requested."
        )

    sampled_players.extend(random.sample(candidates, PLAYERS_PER_GROUP))

players_df = pd.DataFrame(sampled_players)
players_df.insert(0, "player_id", range(1, len(players_df) + 1))
players_df["game_name"] = None
players_df["tag_line"] = None
players_df


### Check the sampled rank composition

In [ ]:
players_df.groupby("rank_group").size()


## 6. Match-V5 helpers

The match-history request is filtered to queue `420`, so we request Ranked Solo/Duo matches directly instead of downloading mixed histories and filtering afterward.

`get_match()` caches every full match response as JSON. Shared matches and reruns load from disk instead of downloading the same match again.


In [ ]:
def get_ranked_match_ids(puuid, count=RANKED_MATCHES_PER_PLAYER):
    url = f"{REGION_BASE}/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params = {
        "start": 0,
        "count": count,
        "queue": RANKED_SOLO_QUEUE_ID,
    }
    return riot_get(url, params=params)


def get_match(match_id):
    cache_file = CACHE_DIR / f"{match_id}.json"

    if cache_file.exists():
        with cache_file.open("r", encoding="utf-8") as file:
            return json.load(file), True

    url = f"{REGION_BASE}/lol/match/v5/matches/{match_id}"
    match = riot_get(url)

    with cache_file.open("w", encoding="utf-8") as file:
        json.dump(match, file)

    return match, False


## 7. Fetch ranked match IDs for every sampled player

This stage requests only ID lists, not full match details.


In [ ]:
player_match_ids = {}
history_failures = []

for i, player in players_df.iterrows():
    puuid = player["puuid"]

    try:
        ids = get_ranked_match_ids(puuid)
        player_match_ids[puuid] = ids
        print(f"[{i + 1:02d}/{len(players_df)}] {player['rank_group']:10s} — {len(ids)} ranked match IDs")

    except Exception as exc:
        history_failures.append({"puuid": puuid, "error": str(exc)})
        player_match_ids[puuid] = []
        print(f"[{i + 1:02d}/{len(players_df)}] FAILED — {exc}")

print()
print("History failures:", len(history_failures))


## 8. Fetch match details and build the player-match dataset

A match shared by multiple sampled players creates one analytical row per sampled player, but its full JSON is downloaded only once.

Riot ID (`game_name`, `tag_line`) is recovered from the participant record, avoiding an extra Account-V1 request for every sampled player.


In [ ]:
rows = []
match_failures = []
network_downloads = 0
cache_hits = 0

player_lookup = players_df.set_index("puuid").to_dict("index")

for player_number, (puuid, match_ids) in enumerate(player_match_ids.items(), start=1):
    player_info = player_lookup[puuid]
    print(f"\nPlayer {player_number:02d}/{len(player_match_ids)} ({player_info['rank_group']})")

    for j, match_id in enumerate(match_ids, start=1):
        try:
            match, from_cache = get_match(match_id)

            if from_cache:
                cache_hits += 1
            else:
                network_downloads += 1

            info = match["info"]
            if info.get("queueId") != RANKED_SOLO_QUEUE_ID:
                continue

            participant = next(p for p in info["participants"] if p["puuid"] == puuid)
            duration_min = info["gameDuration"] / 60
            cs = participant["totalMinionsKilled"] + participant["neutralMinionsKilled"]

            rows.append({
                "player_id": player_info["player_id"],
                "puuid": puuid,
                "rank_group": player_info["rank_group"],
                "sample_tier": player_info["tier"],
                "sample_division": player_info["division"],
                "sample_league_points": player_info["league_points"],
                "game_name": participant.get("riotIdGameName"),
                "tag_line": participant.get("riotIdTagline"),
                "match_id": match_id,
                "game_start": pd.to_datetime(info["gameStartTimestamp"], unit="ms", utc=True),
                "queue_id": info["queueId"],
                "game_mode": info.get("gameMode"),
                "game_duration_min": duration_min,
                "champion": participant["championName"],
                "position": participant.get("teamPosition"),
                "win": participant["win"],
                "kills": participant["kills"],
                "deaths": participant["deaths"],
                "assists": participant["assists"],
                "kda": (participant["kills"] + participant["assists"]) / max(participant["deaths"], 1),
                "gold": participant["goldEarned"],
                "damage_to_champions": participant["totalDamageDealtToChampions"],
                "damage_taken": participant["totalDamageTaken"],
                "vision_score": participant["visionScore"],
                "cs": cs,
                "cs_per_min": cs / duration_min if duration_min > 0 else None,
            })

            if j % 20 == 0 or j == len(match_ids):
                print(f"  processed {j:3d}/{len(match_ids)}")

        except Exception as exc:
            match_failures.append({
                "puuid": puuid,
                "match_id": match_id,
                "error": str(exc),
            })

print()
print("New full-match downloads:", network_downloads)
print("Full-match cache hits:", cache_hits)
print("Match failures:", len(match_failures))


## 9. Build and inspect the final match-history table

In [ ]:
matches_df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["puuid", "match_id"])
    .sort_values(["rank_group", "player_id", "game_start"])
    .reset_index(drop=True)
)

print("Player-match rows:", len(matches_df))
print("Unique sampled players:", matches_df["puuid"].nunique())
print("Unique Riot matches:", matches_df["match_id"].nunique())

matches_df.head()


### Match-history coverage per player

In [ ]:
coverage = (
    matches_df
    .groupby(["player_id", "rank_group", "puuid"])
    .agg(
        ranked_matches=("match_id", "count"),
        first_match=("game_start", "min"),
        last_match=("game_start", "max"),
    )
    .reset_index()
)
coverage


### Coverage by rank group

In [ ]:
coverage.groupby("rank_group")["ranked_matches"].agg(
    ["count", "min", "median", "mean", "max"]
)


## 10. Fill Riot IDs in the player table

Riot IDs found in collected match histories are copied back into `players_df`. PUUID remains the stable player key.


In [ ]:
riot_ids = (
    matches_df
    .dropna(subset=["game_name"])
    .sort_values("game_start")
    .groupby("puuid", as_index=False)
    .last()[["puuid", "game_name", "tag_line"]]
)

players_df = (
    players_df
    .drop(columns=["game_name", "tag_line"])
    .merge(riot_ids, on="puuid", how="left")
)

players_df


## 11. Save outputs for Notebook 02

Notebook 02 can load the files with:

```python
players_df = pd.read_csv("data/processed/players.csv")
matches_df = pd.read_csv(
    "data/processed/ranked_match_history.csv",
    parse_dates=["game_start"]
)
```

The raw match cache remains in `data/cache/matches/`.


In [ ]:
players_path = PROCESSED_DIR / "players.csv"
matches_path = PROCESSED_DIR / "ranked_match_history.csv"

players_df.to_csv(players_path, index=False)
matches_df.to_csv(matches_path, index=False)

print("Saved:")
print(" ", players_path.resolve())
print(" ", matches_path.resolve())


## 12. Optional failure inspection

In [ ]:
print("History failures:")
display(pd.DataFrame(history_failures))

print("\nMatch failures:")
display(pd.DataFrame(match_failures))
